<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Solution: Anomaly-based inspection

**[Anomaly-based inspection](https://learning.nextia-ai.com/courses/vision/m06/anomaly-based-inspection/)** · Vision AI: See, Count and Inspect · Module 6, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** score returned candles by their patches, build a small PatchCore by hand, run anomalib's PatchCore and PaDiM on good photos only, read a heat map, and compare the inspectors by AUROC and by Hamid's cost per day.

| | |
|---|---|
| **Time** | About 45 minutes. anomalib's PatchCore learns for about 2 minutes on a laptop CPU, and longer on a free Colab or Kaggle CPU. |
| **Needs** | An internet connection for the setup cells and the models (ResNet-18 47 MB, wide ResNet-50 about 270 MB). No account and no key. |
| **You should know** | The two ways to find damage and AUROC, from [Two ways to find damage](https://learning.nextia-ai.com/courses/vision/m06/two-ways-to-find-damage/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 with PyTorch 2.15.0, timm 1.0.31 and anomalib 2.7.0, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m06/anomaly-based-inspection/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M06-L02-anomaly-based-inspection/anomaly-based-inspection-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, torch 2.15.0, torchvision 0.29.2, timm 1.0.31, anomalib 2.7.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2 torch==2.15.0 torchvision==0.29.2 timm==1.0.31 anomalib==2.7.0
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)
import torch; print("torch", torch.__version__)
import torchvision; print("torchvision", torchvision.__version__)
import timm; print("timm", timm.__version__)
import anomalib; print("anomalib", anomalib.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 19 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/returns.zip": "dcf02f521f68228d34717b303628b2db8b19d18433126da2ce64047f1449210f",
    "archives/extras.zip": "f121137bc765866c25929a4233c557ba72b64eea0c806070cc04a86c28ec27ec",
    "compare.py": "e9794acacbce00c240906cc8b3e2256b8505ff4a8074b7507a8b159cd39b4826",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Patches, not the whole photo

In the last lesson, one embedding for the whole photo gave an AUROC of only 0.73: a small spot is lost among four candles. Here, we keep the middle layers of ResNet-18. They give one list of 384 numbers for each small square of the photo: a **patch**. A photo of 224 x 224 pixels gives a grid of 28 x 28 = 784 patches.

In [ ]:
returns = pd.read_csv("data/returns/manifest.csv")
train, val, test = (returns[returns["split"] == s].reset_index(drop=True) for s in ("train", "val", "test"))
good_train = train[train["status"] == "good"]
dval, dte = (val["status"] == "damaged").values, (test["status"] == "damaged").values

import torch, timm
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
torch.set_num_threads(4)
# ResNet-18 trained on ImageNet (timm, Apache-2.0 weights), at a fixed revision, as in the embeddings lesson.
# features_only: the network stops early and gives its middle layers, which still know WHERE things are.
weights = hf_hub_download("timm/resnet18.a1_in1k", "model.safetensors", revision="491b427b45c94c7fb0e78b5474cc919aff584bbf")
backbone = timm.create_model("resnet18.a1_in1k", pretrained=True, features_only=True, out_indices=(2, 3),
                             pretrained_cfg_overlay={"file": weights}).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def patches(path):
    """784 patches (a 28 x 28 grid) of 384 numbers each, for one photo resized to 224 x 224."""
    a = np.asarray(Image.open(path).convert("RGB").resize((224, 224), Image.BILINEAR), np.float32) / 255
    x = ((torch.from_numpy(a).permute(2, 0, 1) - MEAN) / STD)[None]
    with torch.no_grad():
        f2, f3 = backbone(x)                       # 128 x 28 x 28 and 256 x 14 x 14
        f2 = F.avg_pool2d(f2, 3, 1, 1)             # each patch also sees its neighbours
        f3 = F.interpolate(F.avg_pool2d(f3, 3, 1, 1), size=f2.shape[-2:], mode="bilinear", align_corners=False)
        f = torch.cat([f2, f3], 1)[0]              # 384 x 28 x 28
    return f.permute(1, 2, 0).reshape(-1, f.shape[0]).numpy()

p = patches("data/" + good_train["file"].iloc[0])
print(p.shape)

## 2. The memory of good patches

We take the patches of the 380 good training photos and keep 20,000 of them at random. This is the **memory**: what good wax, good wicks and good background look like. It takes about 20 seconds.

In [ ]:
bank = np.vstack([patches("data/" + f) for f in good_train["file"]])
rng = np.random.default_rng(0)
memory = bank[rng.choice(len(bank), 20000, replace=False)].astype(np.float32)
memory_sq = (memory ** 2).sum(1)
print(f"{len(bank):,} good patches, {len(memory):,} kept in the memory")

def patch_distances(path):
    """For each of the 784 patches: the distance to the nearest patch in the memory of good patches."""
    q = patches(path)
    d2 = (q ** 2).sum(1)[:, None] + memory_sq[None] - 2 * q @ memory.T
    return np.sqrt(np.maximum(d2.min(1), 0))

## 3. A heat map: the distance of every patch

For each patch of a damaged test photo, we find the nearest good patch in the memory. A patch of normal wax has a close neighbour. A patch of damage does not. Drawn as a 28 x 28 picture, the distances are a **heat map**.

> **Predict.** Where will the heat map be brightest?

In [ ]:
one = test[test["item_id"] == "R-33640732"].iloc[0]
dist = patch_distances("data/" + one["file"])
fig, ax = plt.subplots(1, 3, figsize=(10, 3.2))
ax[0].imshow(Image.open("data/" + one["file"])); ax[0].set_title(one["damage"], fontsize=9)
ax[1].imshow(Image.open("data/" + one["mask_file"])); ax[1].set_title("true damage (mask)", fontsize=9)
ax[2].imshow(dist.reshape(28, 28), cmap="inferno"); ax[2].set_title("patch distances", fontsize=9)
for a in ax: a.axis("off")
plt.show()
print(f"largest patch distance {dist.max():.2f}, median {np.median(dist):.2f}")

## 4. One score per photo: the largest patch distance

A photo is as odd as its oddest patch, so the photo's score is the **largest** patch distance. Scoring the 330 validation and test photos takes about 15 seconds.

In [ ]:
small_val = np.array([patch_distances("data/" + f).max() for f in val["file"]])
small_test_dist = [patch_distances("data/" + f) for f in test["file"]]
small_test = np.array([d.max() for d in small_test_dist])
print("small PatchCore, AUROC on test:", cp.roc_auc(small_test, dte))

> **Check.** You should see 0.847. This is the project's small PatchCore: the image service of Module 8 runs the same idea (made smaller and faster, it gets 0.836).

A common mistake: these scores are distances, not probabilities. They go from about 5 to 19. A line at 0.5, as for a probability, fails every photo:

In [ ]:
print(f"scores from {small_test.min():.2f} to {small_test.max():.2f}")
print(cp.inspection_costs(small_test, dte, 0.5, 0.5))

## 5. anomalib's PatchCore with a wide ResNet-50

[anomalib](https://github.com/open-edge-platform/anomalib) (Apache-2.0) has PatchCore and PaDiM ready to use. Its PatchCore uses a bigger backbone, the wide ResNet-50, and chooses the patches of its memory with care (a **coreset**: patches that cover all the kinds of good patch, not a random sample). anomalib reads photos from folders, so we copy the good training photos into one folder and the damaged ones into another (anomalib uses the damaged ones only to scale its scores to 0–1).

This cell downloads the wide ResNet-50 (about 270 MB) and learns for about 2 minutes on a laptop CPU.

In [ ]:
import logging, shutil
from pathlib import Path
for name in ("lightning", "lightning.pytorch", "lightning.fabric", "anomalib"):
    logging.getLogger(name).setLevel(logging.ERROR)
from anomalib.data import Folder, PredictDataset
from anomalib.engine import Engine
from anomalib.models import Patchcore, Padim

work = Path("anomalib")
for sub, part in (("train_good", train[train["status"] == "good"]), ("train_damaged", train[train["status"] == "damaged"])):
    (work / sub).mkdir(parents=True, exist_ok=True)
    for f in part["file"]:
        shutil.copy("data/" + f, work / sub)
for split, part in (("val", val), ("test", test)):
    (work / split).mkdir(parents=True, exist_ok=True)
    for f in part["file"]:
        shutil.copy("data/" + f, work / split)

def fit_and_score(model):
    """Learn from the good photos, then score the validation and test photos. Returns two arrays and the heat maps."""
    folder = Folder(name="candles", root=work, normal_dir="train_good", abnormal_dir="train_damaged",
                    num_workers=0, train_batch_size=16, eval_batch_size=16, seed=0)
    engine = Engine(accelerator="cpu", devices=1, logger=False, default_root_dir=work / "results",
                    enable_progress_bar=False)
    engine.fit(model=model, datamodule=folder)
    out, maps = {}, {}
    for split in ("val", "test"):
        for batch in engine.predict(model=model, dataset=PredictDataset(path=work / split, image_size=(256, 256)),
                                    return_predictions=True):
            for i, path in enumerate(batch.image_path):
                out[Path(path).stem] = float(batch.pred_score.flatten()[i])
                maps[Path(path).stem] = batch.anomaly_map[i].squeeze().numpy()
    return np.array([out[i] for i in val["item_id"]]), np.array([out[i] for i in test["item_id"]]), maps

import time
start = time.time()
pc_val, pc_test, pc_maps = fit_and_score(Patchcore(backbone="wide_resnet50_2", coreset_sampling_ratio=0.01, visualizer=False))
print(f"PatchCore, wide ResNet-50: {time.time() - start:.0f} s, AUROC on test: {cp.roc_auc(pc_test, dte)}")

In [ ]:
expect('the PatchCore AUROC (about 0.917; your run may differ a little)', cp.roc_auc(pc_test, dte), 0.917, tolerance=0.04)

anomalib's result is not exactly the same from run to run: the course's recorded run gave 0.917, and five runs of this notebook on one laptop gave between 0.92 and 0.94. The check accepts a small difference.

Its heat map for the same candle:

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3.2))
ax[0].imshow(Image.open("data/" + one["file"])); ax[0].set_title(one["damage"], fontsize=9)
ax[1].imshow(pc_maps[one["item_id"]], cmap="inferno"); ax[1].set_title("PatchCore heat map", fontsize=9)
for a in ax: a.axis("off")
plt.show()

## 6. PaDiM: one bell curve per patch position

PaDiM learns, for each **position** in the grid, the average good patch and how much good patches vary there. A patch far from its position's average is odd. It learns in seconds, because it keeps averages instead of a memory.

In [ ]:
start = time.time()
pd_val, pd_test, _ = fit_and_score(Padim(backbone="resnet18", visualizer=False))
print(f"PaDiM, ResNet-18: {time.time() - start:.0f} s, AUROC on test: {cp.roc_auc(pd_test, dte)}")

## 7. Hamid's cost per day

As in the last lesson, `cp.inspection_row` chooses the pass, check and fail lines on the validation photos (at most 100 checks a day for 600 returns) and measures the cost on the test photos.

In [ ]:
rows = [cp.inspection_row(name, v, dval, t, dte, list(test["damage"]), "returns test", None, None, 380, "Apache-2.0")
        for name, v, t in (("small PatchCore (ResNet-18, by hand)", small_val, small_test),
                           ("PatchCore, wide ResNet-50 (anomalib)", pc_val, pc_test),
                           ("PaDiM, ResNet-18 (anomalib)", pd_val, pd_test))]
for r in rows:
    print(f"{r['method']:38s} cost {r['value']:7.1f} a day | AUROC {r['auroc']} | worst {r['worst']} {r['worst_value']}")

Compare with the course's recorded rows: small PatchCore 1290.8 dollars a day, PatchCore with a wide ResNet-50 368.8, PaDiM 993.6, and no inspection 2229.4. With so few damaged photos, one photo more or less on the wrong side of a line moves the cost a lot.

## 8. Your turn: the mean instead of the largest distance

> **Your turn.** Score each test photo by the **mean** of its patch distances instead of the largest one, and put the AUROC on the test photos in `mean_auc`. Use `small_test_dist` (the patch distances of every test photo) and `cp.roc_auc`. Predict first: higher or lower than 0.847?

In [ ]:
mean_auc = cp.roc_auc([d.mean() for d in small_test_dist], dte)
print(mean_auc)

In [ ]:
expect('mean_auc', mean_auc, 0.875, tolerance=0.005)

Did you predict lower? We did too: a small damage covers a few of the 784 patches, and the other patches pull the mean back to normal. On these 18 damaged test photos, the mean did **better** (0.875 against 0.847). Several of them have large damage, such as wax melted out of the candle, which raises many patches at once. A rule that sounds right must still be measured, and with only 18 damaged photos, a difference this small could change with other photos. Choose such a rule on the validation photos, never on the test photos.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m06/anomaly-based-inspection/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Set the line with the warehouse](https://learning.nextia-ai.com/courses/vision/m06/set-the-line-with-the-warehouse/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m06/anomaly-based-inspection/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.